The plan for this file it to have all the code to go from the different planet spectra to them making a new planet list hopefully in an organsied way so it can all be done from one file. 


In [ ]:
#--- Imports ---#
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd
from uncertainties import  ufloat
from scipy import constants
from pathlib import Path
from typing import Tuple, List
from functools import partial
from time import time
import concurrent.futures
from tqdm import tqdm
import os
from datetime import datetime 

#--- Data sheets ---# 

Planet_data= pd.read_csv('Data/planets.csv', comment='#')



# --- Classes ---#

class Configuration:








In [ ]:
# -----------------------------
# 2. Model + likelihood
# -----------------------------

def quadratic_model(T, A, B, C):
    return A*T**2 + B*T + C
    
def log_likelihood(theta, T, y, yerr):
    A, B, C = theta
    model = quadratic_model(T, A, B, C)
    return -0.5 * np.sum(
        (y - model)**2 / yerr**2 + np.log(2*np.pi*yerr**2)
    )
    
def log_prior(theta):
    A, B, C = theta
    if -30 < A < 30 and -30 < B < 30 and 0 < C < 40:
        return 0.0
    return -np.inf

def log_probability(theta, T, y, yerr):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, T, y, yerr)

# -----------------------------
# 3. MCMC runner
# -----------------------------

def run_mcmc(T, y, yerr, nsteps=9000):
    ndim, nwalkers = 3, 64

    pos = np.zeros((nwalkers, ndim))
    pos[:, 0] = np.random.uniform(-1e-2, 1e-2, nwalkers)   # A
    pos[:, 1] = np.random.uniform(-3, 3, nwalkers)        # B
    pos[:, 2] = np.random.normal(np.mean(y), 0.2*np.std(y), nwalkers)

    sampler = emcee.EnsembleSampler(
        nwalkers, ndim, log_probability, args=(T, y, yerr)
    )
    sampler.run_mcmc(pos, nsteps, progress=True)

    burn = 1000
    tau_quad = sampler.get_autocorr_time(discard=burn, thin=1)
    samples = sampler.get_chain(discard=1000, thin=1, flat=True)
    return samples, tau_quad

# -----------------------------
# 7. Linear model + BIC comparison
# -----------------------------

def linear_model(T, B, C):
    return B*T + C


def log_likelihood_linear(theta, T, y, yerr):
    B, C = theta
    model = linear_model(T, B, C)
    return -0.5 * np.sum(
        (y - model)**2 / yerr**2 + np.log(2*np.pi*yerr**2)
    )


def log_prior_linear(theta):
    B, C = theta
    if -20 < B < 20 and -20 < C < 50:
        return 0.0
    return -np.inf


def log_probability_linear(theta, T, y, yerr):
    lp = log_prior_linear(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood_linear(theta, T, y, yerr)


def run_mcmc_linear(T, y, yerr, nsteps=9000):
    ndim, nwalkers = 2, 64

    pos = np.zeros((nwalkers, ndim))
    pos[:, 0] = -2.0 + 1e-3*np.random.randn(nwalkers)   # B
    pos[:, 1] = np.random.uniform(5, 45, nwalkers)  # C

    sampler = emcee.EnsembleSampler(
        nwalkers, ndim, log_probability_linear, args=(T, y, yerr)
    )
    sampler.run_mcmc(pos, nsteps, progress=False)
    
    burn=1000
    tau_lin = sampler.get_autocorr_time(discard=burn, thin=1)
    samples = sampler.get_chain(discard=1000, thin=1, flat=True)
    return samples, tau_lin


#center T values for better convergence
T0 = np.mean(Teq)
Tscale = np.std(Teq)
Tc = (Teq - T0) / Tscale
print(Tc.min(), Tc.max())


#Run MCMC for Quadratic

samples_quad, tau_quad = run_mcmc(
    T=Tc,
    y=feature_height,
    yerr=feature_height_err
)
#Run MCMC for Linear
samples_lin, tau_lin = run_mcmc_linear(
    T=Tc,
    y=feature_height,
    yerr=feature_height_err
)

print("Quadratric Autocorrelation times:", tau_quad)
print("Quadratic Max τ:", np.max(tau_quad))

print("Linear Autocorrelation times:", tau_lin)
print("Linear Max τ:", np.max(tau_lin))

import corner as corner
import matplotlib.pyplot as plt

# -----------------------------
# Quadratic corner plot
# -----------------------------

labels_quad = [
    r"$A$ (curvature)",
    r"$B$ (linear term)",
    r"$C$ (offset)"
]

fig_quad = corner.corner(
    samples_quad,
    labels=labels_quad,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt=".3g",
    title_kwargs={"fontsize": 12}
)

fig_quad.suptitle("Quadratic model posterior", fontsize=14)
plt.show()

# -----------------------------
# Linear corner plot
# -----------------------------

labels_lin = [
    r"$B$ (slope)",
    r"$C$ (offset)"
]

fig_lin = corner.corner(
    samples_lin,
    labels=labels_lin,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt=".3g",
    title_kwargs={"fontsize": 12}
)

fig_lin.suptitle("Linear model posterior", fontsize=14)
plt.show()

#Revert coefficients to uncentered T for easier interpretation

def revert_quadratic_from_centered(samples, T0, Tscale):
    """
    Convert quadratic coefficients fitted in centered+scaled temperature
    back to raw temperature coefficients.

    Tc = (T - T0) / Tscale
    y  = Ac * Tc^2 + Bc * Tc + Cc

    Returns coefficients A, B, C such that:
    y = A * T^2 + B * T + C
    """

    Ac = samples[:, 0]
    Bc = samples[:, 1]
    Cc = samples[:, 2]

    A = Ac / Tscale**2
    B = Bc / Tscale - 2.0 * Ac * T0 / Tscale**2
    C = Cc - Bc * T0 / Tscale + Ac * T0**2 / Tscale**2

    return np.column_stack([A, B, C])
    
samples_quad_uncentered = revert_quadratic_from_centered(
    samples_quad, T0, Tscale
)

def revert_linear_from_centered(samples, T0, Tscale):
    Bc = samples[:, 0]
    Cc = samples[:, 1]

    B = Bc / Tscale
    C = Cc - Bc * T0 / Tscale

    return np.column_stack([B, C])
    
samples_lin_uncentered = revert_linear_from_centered(
    samples_lin,
    T0,
    Tscale
)

# -----------------------------
# 8. Compute BICs
# -----------------------------

# Quadratic fit
A_med, B_med, C_med = np.median(samples_quad_uncentered, axis=0)
logL_quad = log_likelihood(
    [A_med, B_med, C_med],
    Teq,
    feature_height,
    feature_height_err
)

k_quad = 3
n = len(Teq)
BIC_quad = k_quad * np.log(n) - 2 * logL_quad

# Linear fit
B_lin, C_lin = np.median(samples_lin_uncentered, axis=0)

logL_lin = log_likelihood_linear(
    [B_lin, C_lin],
    Teq,
    feature_height,
    feature_height_err
)

k_lin = 2
BIC_lin = k_lin * np.log(n) - 2 * logL_lin

# -----------------------------
# 9. Report comparison
# -----------------------------

delta_BIC = BIC_quad - BIC_lin

print(f"BIC (quadratic) = {BIC_quad:.2f}")
print(f"BIC (linear)    = {BIC_lin:.2f}")
print(f"ΔBIC (quad − lin) = {delta_BIC:.2f}")

if delta_BIC < -10:
    verdict = "Very strong evidence for curvature (quadratic)"
elif delta_BIC < -6:
    verdict = "Strong evidence for curvature"
elif delta_BIC < -2:
    verdict = "Positive evidence for curvature"
elif delta_BIC < 2:
    verdict = "No meaningful preference"
else:
    verdict = "Linear model preferred"

print("Interpretation:", verdict)

#Plotting the linear fit with posterior samples

B_med, C_med = np.median(samples_lin_uncentered, axis=0)
B_err, C_err = np.std(samples_lin_uncentered, axis=0)

print(f"B = {B_med:.4f} ± {B_err:.4f}")
print(f"C = {C_med:.2f} ± {C_err:.2f}")

T_plot = np.linspace(Teq.min(), Teq.max(), 300)

plt.errorbar(
    Teq, feature_height, yerr=feature_height_err,
    fmt='o', color='black'
)

# Posterior cloud
idx = np.random.choice(len(samples_lin_uncentered), 100, replace=False)
for i in idx:
    B, C = samples_lin_uncentered[i]
    plt.plot(
        T_plot,
        B*T_plot + C,
        color='orange',
        alpha=0.15
    )

# Posterior median
plt.plot(
    T_plot,
    B_med*T_plot + C_med,
    color='blue', lw=2.5, label='Posterior median'
)
#plt.ylim(-300,-400)
plt.xlabel("Temperature")
plt.ylabel("Feature height")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

#Plotting Quadratic fit with posterior samples

# Median and 1σ from posterior
A_med, B_med, C_med = np.median(samples_quad_uncentered, axis=0)
A_err, B_err, C_err = np.std(samples_quad_uncentered, axis=0)

print(f"A = {A_med:.6f} ± {A_err:.6f}")
print(f"B = {B_med:.4f} ± {B_err:.4f}")
print(f"C = {C_med:.2f} ± {C_err:.2f}")

T_plot = np.linspace(Teq.min(), Teq.max(), 300)

plt.figure()

# Data with error bars
plt.errorbar(
    Teq,
    feature_height,
    yerr=feature_height_err,
    fmt='o',
    color='black'
)

# Posterior cloud
idx = np.random.choice(len(samples_quad_uncentered), 100, replace=False)
for i in idx:
    A, B, C = samples_quad_uncentered[i]
    plt.plot(
        T_plot,
        A*T_plot**2 + B*T_plot + C,
        color='orange',
        alpha=0.15
    )

# Posterior median curve
plt.plot(
    T_plot,
    A_med*T_plot**2 + B_med*T_plot + C_med,
    color='blue',
    lw=2.5,
    label='Posterior median'
)

plt.xlabel("Temperature")
plt.ylabel("Feature height")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


import numpy as np

# ============================================================
# Constraining power (CENTERED temperature compatible)
# ============================================================

def rank_constraining_points_linear(Tc, yerr):
    """
    Rank points by how much they constrain the linear slope.
    Tc must be CENTERED temperature (Tc = T - <T>).
    """
    weights = 1.0 / yerr**2

    # Pivot is zero by construction for centered T
    T_pivot = 0.0

    power = weights * Tc**2
    order = np.argsort(power)[::-1]

    return order, power, T_pivot


def rank_constraining_points_quadratic(Tc, yerr):
    """
    Rank points by how much they constrain the quadratic curvature.
    Tc must be CENTERED temperature (Tc = T - <T>).
    """
    weights = 1.0 / yerr**2

    # Curvature information scales as Tc^4
    power = weights * Tc**4
    order = np.argsort(power)[::-1]

    return order, power


# ============================================================
# Usage
# ============================================================

# Center temperature (should already exist, but safe)
T0 = np.mean(Teq)
Tc = Teq - T0

# Linear constraining power
order_l, power_l, T_pivot = rank_constraining_points_linear(
    Tc, feature_height_err
)

# Quadratic constraining power
order_q, power_q = rank_constraining_points_quadratic(
    Tc, feature_height_err
)


# ============================================================
# Reporting
# ============================================================

print("===== TOP 5 CONSTRAINING POINTS (LINEAR SLOPE) =====")
print(f"Pivot temperature (Teq): {T0:.1f}")
for i in order_l[:5]:
    print(
        f"T = {Teq[i]:.1f}, "
        f"yerr = {feature_height_err[i]:.2f}, "
        f"power = {power_l[i]:.3e}"
    )

print("\n===== TOP 5 CONSTRAINING POINTS (QUADRATIC CURVATURE) =====")
for i in order_q[:5]:
    print(
        f"T = {Teq[i]:.1f}, "
        f"yerr = {feature_height_err[i]:.2f}, "
        f"power = {power_q[i]:.3e}"
    )




